In [1]:
from pathlib import Path
from typing import Dict
from tqdm import tqdm
import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import umap.umap_ as umap
from scipy.spatial.distance import cosine

from joblib import Parallel, delayed

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.ealstm import EALSTM
from neuralhydrology.nh_run import start_run
from neuralhydrology.utils.config import Config

/storage/vast-gfz-hpc-01/home/wuhlmann/miniforge3/envs/lamah-ce_lstm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!nvidia-smi


Thu Dec  4 16:26:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:CA:00.0 Off |                    0 |
| N/A   30C    P0             79W /  500W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
run_dir = Path("/home/wuhlmann/BA/test_runs/runs/512_batch_size_2711_095750")
cfg = Config(run_dir / "config.yml")

## Initialize model with base paramters from config. 
#cfg = Config(run_dir / "config.yml")
#ea_lstm = EALSTM(cfg=cfg)

## Load the trained weights into the model. 
#weights_path = run_dir / "model_epoch030.pt"
#weights = torch.load(str(weights_path), map_location="cuda:0")
#ea_lstm.load_state_dict(weights)

## Set to eval, to deactivate dropout.
#ea_lstm.to(cfg.device)
#ea_lstm.eval()


pass scaler, so scaler from training is used 

go through every phase and extract cell states per basin, calculate average and save 

In [4]:
def get_basin_ids_from_txt(
    id_txt_path: Path
) -> list[str]: 

    with open(id_txt_path, "r") as f:
        ids_list = f.read().split("\n")
    
    return ids_list

In [5]:
ids = get_basin_ids_from_txt(Path("/home/wuhlmann/BA/data/processed_data/test_splits/test_basin_ids.txt"))

In [6]:
cfg.test_basin_file

PosixPath('/home/wuhlmann/BA/data/processed_data/test_splits/test_basin_ids.txt')

In [7]:
def extract_state_per_basin( 
    run_dir: Path, 
    cfg: Config, 
    scaler,
    basin_id: str, 
    gpu_id: int
) -> tuple[str, dict]: 
    """Parallalizable function, to extract cell state and average it. 

    Parameters
    ----------
    model : EALSTM
        _description_
    config : Config
        _description_
    scaler : _type_
        _description_
    basin_id : str
        _description_

    Returns
    -------
    ds_output
        _description_
    """
    basin_dict = {
        "c_mean": None,
        "c_last": None
    }

    device = torch.device(f"cuda:{gpu_id}")

    # Initialize model with base paramters from config. 
    ea_lstm = EALSTM(cfg=cfg)

    # Load the trained weights into the model. 
    weights_path = run_dir / "model_epoch030.pt"
    weights = torch.load(str(weights_path), map_location=device)
    ea_lstm.load_state_dict(weights)

    # Set to eval, to deactivate dropout.
    ea_lstm.to(device)
    ea_lstm.eval()

    # load dataset for basin_id 
    ds = get_dataset(cfg=cfg, is_train=False, period="test", scaler=scaler, basin=basin_id)
 
    # Do the entire basin at once.
    dataloader = DataLoader(ds, batch_size=15000, shuffle=False, collate_fn=ds.collate_fn, num_workers=0, pin_memory=True)

    with torch.no_grad():
        for data in dataloader: 

            # Move batch to device
            for key in data.keys():
                    if key.startswith('x_d'):
                        data[key] = {k: v.to(device) for k, v in data[key].items()}
                    elif not key.startswith('date'):
                        data[key] = data[key].to(device)

            model_output = ea_lstm(data) 

            # c_n has shape num_samples * 365 * 256
            c_n = model_output["c_n"].cpu().numpy()
 
            # Only take the last day of the prediction for visualisation with shape num_samples * 256
            basin_dict["c_last"] = c_n[:,-1,:]

            # vstack to get a long array with shape (num_samples * 365) * 256 and remove nan from warmup
            c_n = np.vstack(c_n)
            c_n = c_n[~np.isnan(c_n).any(axis=1)]

            # Calculate average cell state over all states occupied. 
            basin_dict["c_mean"] = c_n.mean(axis=0)

            # Free memory 
            del c_n
            del model_output 
            del data
            torch.cuda.empty_cache()

    return basin_id, basin_dict

In [15]:
scaler = load_scaler(run_dir=run_dir)

basin_ids = [
"4",
"18",
"23",
"27",
"32",
"85",
"87",
"91",
"96",
"116",
"117",
"126",
"128",
"138",
"144",
"153",
"157",
"166",
"171",
"176",
"192",
"198",
"205",
"206",
"213",
"219",
"244",
"245",
"254",
"263",
"265",
"274",
"279",
"287",
"295",
"306",
"328",
"333",
"345",
"368",
"371",
"387",
"388",
"402",
"406",
"426",
"442",
"457",
"461",
"473",
"499",
"517",
"518",
"539",
"545",
"548",
"570",
"576",
"582",
"590",
"591",
"602",
"614",
"615",
"622",
"629",
"638",
"639",
"646",
"651",
"655",
"656",
"704",
"706",
"741",
"747",
"748",
"753",
"767",
"791",
"798",
"800",
"802",
"809",
"816",
"832",
"841",
"845"
]

#half = len(basin_ids) // 2

#result_0 = []

#for basin_id in basin_ids[:half]:
    #result_0.append(extract_state_per_basin(run_dir=run_dir, cfg=cfg, scaler=scaler, basin_id=basin_id, gpu_id=0))

parallel_extractor = Parallel(n_jobs=3, verbose=10)

results = parallel_extractor(
    delayed(extract_state_per_basin)(run_dir=run_dir, cfg=cfg,scaler=scaler, basin_id=basin_id, gpu_id=0) 
    for i, basin_id in enumerate(basin_ids)
    )

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done   2 tasks      | elapsed:   15.3s
[Parallel(n_jobs=3)]: Done   7 tasks      | elapsed:   41.5s
[Parallel(n_jobs=3)]: Done  12 tasks      | elapsed:  1.1min
[Parallel(n_jobs=3)]: Done  19 tasks      | elapsed:  1.6min
[Parallel(n_jobs=3)]: Done  26 tasks      | elapsed:  2.2min
[Parallel(n_jobs=3)]: Done  35 tasks      | elapsed:  2.9min
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:  3.5min
[Parallel(n_jobs=3)]: Done  55 tasks      | elapsed:  4.3min
[Parallel(n_jobs=3)]: Done  66 tasks      | elapsed:  5.1min
[Parallel(n_jobs=3)]: Done  79 tasks      | elapsed:  6.0min


KeyboardInterrupt: 

In [ ]:
def get_basin_ids_from_txt(
    id_txt_path: Path
) -> list[str]: 

    with open(id_txt_path, "r") as f:
        ids_list = f.read().split("\n")
            
    return ids_list[:-1]

2 gpu 3 n j = 6.47

In [ ]:
get_basin_ids_from_txt("/home/wuhlmann/BA/data/processed_data/test_splits/train_basin_ids.txt")

In [ ]:
test_state_dict = dict(results)

import pickle

with open(Path("/home/wuhlmann/BA/test_runs/runs/512_batch_size_2711_095750/test/model_epoch030") / "cell_states.p", "wb") as f:
    pickle.dump(test_state_dict, f)

In [ ]:
import pickle

with open("/home/wuhlmann/BA/data/processed_data/cell_states/3phase_test.p", "rb") as f:
    test_state_dict = pickle.load(f)

In [ ]:
len(test_state_dict["1"]["c_last"])

In [ ]:
with open("/home/wuhlmann/BA/data/processed_data/test_splits/strata_ids.txt", "rb") as f: 
    strata_map = pd.read_table(f,header=0, sep=",")

strata_map.set_index("ID", inplace=True)
strata_map.rename(columns={"0":"strata"}, inplace=True)

test_strata = strata_map.iloc[[int(k) for k in test_state_dict],0]

In [ ]:
reducer = umap.UMAP(
    n_components=2, 
    n_neighbors=5,
    min_dist=0.1,
    random_state=42
)

m2d = reducer.fit_transform(np.vstack([test_state_dict[basin_id]["c_mean"] for basin_id in test_state_dict]))


fig, ax = plt.subplots(1,1)

ax.scatter(m2d[:,0],m2d[:,1], c=test_strata)


In [ ]:
test_state_dict["116"]["2d_mean"]

In [ ]:
scaler = load_scaler(run_dir=run_dir)

basin_ids = ["101", "102"]

data_set_101 = get_dataset(cfg=cfg, is_train=False, period="train", scaler=scaler, basin=basin_ids[0])
data_set_102 = get_dataset(cfg=cfg, is_train=False, period="train", scaler=scaler, basin=basin_ids[1])

In [ ]:
ds_output = []

for i, ds in enumerate([data_set_101, data_set_102]): 
    
    dataloader = DataLoader(ds, batch_size=15000, shuffle=False, collate_fn=ds.collate_fn, num_workers=cfg.num_workers, pin_memory=True)
    output = []

    with torch.no_grad():
        for data in tqdm(dataloader): 
            for key in data.keys():
                    if key.startswith('x_d'):
                        data[key] = {k: v.to(cfg.device) for k, v in data[key].items()}
                    elif not key.startswith('date'):
                        data[key] = data[key].to(cfg.device)
            out = ea_lstm(data) 
            c_n = out["c_n"].cpu().numpy()
            
            # Only take the last day of the prediction for raw_
            output.append(c_n)

            del out
            del data
            torch.cuda.empty_cache()

    ds_output.append(output)

In [ ]:
ds_output[1][0].shape

In [ ]:
import numpy as np

ordered_states = []

for output in ds_output:
    states = np.vstack(output)
    states = np.vstack(states[:,:,:])
    print(states.shape)
    states = states[~np.isnan(states).any(axis=1)]
    states.shape

    ordered_states.append(states)

In [ ]:
mean_101 = ordered_states[0].mean(axis=0)
mean_102 = ordered_states[1].mean(axis=0)

print(cosine(mean_101, mean_102))

In [ ]:
mode = "normal"

In [ ]:
normalizer = StandardScaler()

pca = PCA(0.9,svd_solver='full', random_state=42)

pca_101 = pca.fit_transform(normalizer.fit_transform(ordered_states[0]))
pca_102 = pca.transform(normalizer.fit_transform(ordered_states[1]))

reducer = umap.UMAP(
    n_components=2, 
    n_neighbors=100,
    min_dist=0.5,
    random_state=42
)

if mode == "raw": 
    states_101 = reducer.fit_transform(ordered_states[0])
    states_102 = reducer.transform(ordered_states[1])
elif mode == "normal": 
    states_101 = reducer.fit_transform(normalizer.fit_transform(ordered_states[0]))
    states_102 = reducer.fit_transform(normalizer.fit_transform(ordered_states[1]))
elif mode == "pca": 
    states_101 = reducer.fit_transform(pca_101)
    states_102 = reducer.transform(pca_102)



In [ ]:
fig, axs = plt.subplots(1,2, figsize=(10,5))

for i,ds in enumerate([states_101, states_102]):

    idx = np.arange(len(ds))
    axs[i].scatter(ds[:365,0],ds[:365,1], cmap="viridis")
    axs[i].title.set_text(f"{basin_ids[i]}")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8,8))

ax.scatter(states_101[:365,0],states_101[:365,1], color="red")
ax.scatter(states_102[:365,0],states_102[:365,1], color="blue")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,projection="3d")

ax.scatter(states_101[:,0], states_101[:,1],states_101[:,2], c=point_colors)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax.view_init(elev=10, azim=45)
plt.show()